# Notebook 5: Convergence Analysis
**Project:** Monte Carlo Risk & Derivatives Pricing
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy import stats
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})
returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
weights = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
port = returns @ weights
print(f'Loaded {len(port):,} observations')


Loaded 2,609 observations


In [2]:
from options_pricing import bs_call
from variance_reduction import mc_naive, mc_antithetic
S0,K,r,T,sigma = 100,100,0.05,1.0,0.20
bs_true = bs_call(S0,K,T,r,sigma)
sizes_c=[100,250,500,1000,2500,5000,10000,25000,50000,100000]
naive_p,anti_p,naive_se,anti_se=[],[],[],[]
for n in sizes_c:
    p_n,se_n=mc_naive(S0,K,T,r,sigma,n)
    p_a,se_a=mc_antithetic(S0,K,T,r,sigma,n)
    naive_p.append(p_n); anti_p.append(p_a)
    naive_se.append(se_n); anti_se.append(se_a)
print(f'BS Exact Price   : ${bs_true:.4f}')
print(f'MC at 100k sims  : ${naive_p[-1]:.4f} (error=${abs(naive_p[-1]-bs_true):.4f})')
print(f'Anti at 100k sims: ${anti_p[-1]:.4f} (error=${abs(anti_p[-1]-bs_true):.4f})')

BS Exact Price   : $10.4506
MC at 100k sims  : $10.4739 (error=$0.0233)
Anti at 100k sims: $10.4529 (error=$0.0023)


In [3]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
ax1=axes[0]
ax1.plot(sizes_c,naive_p,'o-',color='#185FA5',linewidth=1.8,markersize=5,label='Naive MC')
ax1.plot(sizes_c,anti_p,'s-',color='#E24B4A',linewidth=1.8,markersize=5,label='Antithetic')
ax1.axhline(bs_true,color='#1D9E75',linewidth=2,linestyle='--',label=f'BS Exact=${bs_true:.4f}')
ax1.fill_between(sizes_c,[p-2*s for p,s in zip(naive_p,naive_se)],[p+2*s for p,s in zip(naive_p,naive_se)],alpha=0.12,color='#185FA5')
ax1.set_title('Price Convergence: Naive MC vs Antithetic')
ax1.set_xlabel('Simulations'); ax1.set_ylabel('Call Price ($)'); ax1.set_xscale('log'); ax1.legend(fontsize=9)
ax2=axes[1]
ax2.loglog(sizes_c,naive_se,'o-',color='#185FA5',linewidth=1.8,markersize=5,label='Naive MC SE')
ax2.loglog(sizes_c,anti_se,'s-',color='#E24B4A',linewidth=1.8,markersize=5,label='Antithetic SE')
ref=[naive_se[0]*(sizes_c[0]/n)**0.5 for n in sizes_c]
ax2.loglog(sizes_c,ref,'k--',linewidth=1.2,alpha=0.5,label='O(1/sqrt(n)) reference')
ax2.set_title('SE Convergence Rate (log-log)'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/06_convergence_analysis.png',dpi=150,bbox_inches='tight')
plt.show()